# It Moves Now

A floating wind turbine that never moves isn't telling us very much. Part 1 of this
series held the platform at a single frozen offset, a useful first look at the
mooring physics, but no real platform sits still. It surges, heaves, and pitches
continuously under wind and waves, restrained by three mooring lines pulling in three
different directions at once. Does the machine-learning surrogate still keep pace
with the analytical catenary model once the platform is actually moving?

This notebook answers that by rebuilding the floating turbine as a full 6DOF Modelica
MultiBody model, exporting it as an FMU, and driving it from Python under combined
wind and wave loading, with either the analytical catenary or Part 1's ONNX surrogate
computing each line's force every step, reproducing the
[Part 2 blog post](part2-dynamic-mooring.md) live, top to bottom.

![FOWT platform animation](figures/model_gif.gif)

*The platform in motion: OpenModelica's native MultiBody animation, driven by the
same model that produces every number in this notebook.*

In [1]:
from notebook_helpers import *

---

## Setting Up a Three-Line Floating Platform

I keep the same OC4 DeepCwind geometry from Part 1 and extend it to three mooring
lines at 120° spacing: 200 m water depth, 837.6 m anchor radius, 40.868 m fairlead
radius, giving each line a 796.732 m nominal horizontal fairlead-to-anchor offset
(Part 1 used a single line of the same geometry).

Coordinates printed below use the reader-facing plan-view convention
(y = R·sin(azimuth)). The Modelica world frame the code and FMU actually operate
in flips the sway sign, as the plan-view caption notes.

```
Anchor radius:        837.6 m
Fairlead radius:      40.868 m
Anchor depth:         -200.0 m
Fairlead depth (CG):  -14.0 m
Line 1 anchor: (   837.6,     -0.0, -200.0) m
Line 2 anchor: (  -418.8,   -725.4, -200.0) m
Line 3 anchor: (  -418.8,    725.4, -200.0) m
```

| | Hs | Tp | U_hub | TI |
| - | - | - | - | - |
| operational | 2.5 | 10.0 | 12.0 | 0.06 |
| storm | 6.0 | 14.0 | 20.0 | 0.12 |

In [2]:
# Part 1 -- mooring layout, plan view (looking down the heave axis)
theta = np.linspace(0, 2 * np.pi, 200)
fig_layout = go.Figure()
fig_layout.add_trace(go.Scatter(x=R_ANCHOR * np.cos(theta), y=-R_ANCHOR * np.sin(theta),
                                 mode="lines", line={"dash": "dot", "color": "gray"}, name="Anchor radius"))
for i in range(3):
    ax, ay, _ = anchor_position(i)
    fig_layout.add_trace(go.Scatter(x=[0, ax], y=[0, ay], mode="lines+markers",
                                     name=f"Line {i + 1}"))
fig_layout.update_layout(title="3-line mooring layout (plan view)", xaxis_title="x (m)",
                          yaxis_title="y (m)", yaxis_scaleanchor="x", height=450)
fig_layout.show()

*Plan view of the three-line layout: anchors, fairleads, and the nominal 796.7 m
offset circle, all 120° apart. `line_geometry.py` derives the sway component as
`-sin(azimuth)`, matching Modelica's own frame convention for
`mooringLineNRotation` rather than the naive `+sin(azimuth)` a first pass might
reach for.*

I model the platform as a single rigid body riding on six stacked joints: three
prismatic (surge along X, sway along Z, heave along Y), then three revolute (yaw
about Y, roll about X, pitch about Z). Three of those six get a genuine
hydrostatic restoring spring (heave, roll, and pitch), sized directly from the
platform's waterplane geometry:

| DOF | Stiffness |
| - | - |
| Heave | 3 836 kN/m |
| Roll | 1.453×10⁶ kN·m/rad |
| Pitch | 1.453×10⁶ kN·m/rad |

Roll and pitch share the same value by the platform's 3-fold symmetry, and both
come entirely from the waterplane-pressure term, ρgI_waterplane, ≈ 1.453×10⁹
N·m/rad. Heave's spring is hard-coded to the OC4-published hydrostatic
restoring value in heave, C33 = 3 836 kN/m (NREL/TP-5000-60601).

![FOWT top-level diagram](figures/FOWT.png)

*Top-level Modelica diagram exported to `FOWT.fmu`. `platformBody` receives the
three mooring-force vectors, while Python drives `pitchCollective`, `windSpeed`,
`waveForces`, and `waveMoments` every step.*

Rotor aerodynamics run in every simulation here: NREL 5-MW blade geometry, hub-height
wind drawn from the same Kaimal turbulence time series as the sea state, and thrust
from a `Ct(tipSpeedRatio, pitch)` lookup of the real ROSCO rotor-performance table
rather than a closed-form curve. Collective pitch is fixed for each run at that sea
state's published Region-3 operating point (Jonkman et al. 2009, NREL/TP-500-38060,
Table 7-1) -- 3.83° at 12 m/s, 17.47° at 20 m/s -- rather than re-targeted every
step. This model computes no generator power and runs no closed-loop generator-
speed/pitch feedback: rotor speed is itself prescribed (tracking the optimal
tip-speed ratio below rated wind, capped at the rated speed above it), and the pitch
values above are the reference turbine's steady Region-3 operating points, not the
output of a live power-regulation controller. Two sea states drive the loads in
this notebook (the `SEA_STATES` dict above):

| Sea state | Hs | Tp | Wind speed | Turbulence intensity |
| - | - | - | - | - |
| Operational | 2.5 m | 10 s | 12 m/s | 6% |
| Storm | 6.0 m | 14 s | 20 m/s | 12% |

Wave elevation comes from a JONSWAP spectrum (`environment/wave_model.py`) and
feeds `environment/loads.py`, which turns it into a surge force and a pitch
moment applied each step. The FMU accepts a full `waveMoments[3]` vector about
world [X, Y, Z] (roll/yaw/pitch axes); this notebook only excites the Z (pitch)
component, leaving X and Y at zero the same way `waveForces`' sway and heave
components are zero, since a single-heading JONSWAP spectrum gives no roll or
yaw excitation to model. Wind speed feeds the FMU's rotor directly, every step,
driving real aerodynamic thrust through the schedule above. This is wind *and*
wave loading together, not a wave-only simplification.

Each result in this notebook is one deterministic 200 s realization (fixed seed
42 for both wind and wave generation) -- enough for this catenary-vs-surrogate
comparison, but not converged for environmental extremes, so max surge, pitch,
and tension shouldn't be read as design-load estimates. The wave force itself is
a simplified, unvalidated Morison-type proxy (`environment/loads.py`: fixed
`A_wp = 40 m²`, inertial coefficient `1e6`, 10 m pitch-moment arm), not a
validated hydrodynamic model, so absolute platform-response magnitudes here are
illustrative rather than predictive.

---

## Building the Dynamic Co-Simulation Loop

Two implementation details matter when reading the response. Surge and sway
have no hydrostatic restoring spring at all -- the mooring lines supply all of
the horizontal restoring -- only a quadratic viscous damper
(`f = quadraticDragCoefficient·v·|v|`, coefficient 3.95×10⁵ N·s²/m² from
NREL/TP-5000-60601 Table 4-4's published additional-drag term for
potential-flow-only models) to damp the numerically stiff externally-moored
DOFs. Yaw restoring stiffness is zero for the same reason: buoyancy gives no
yaw righting moment, so restoring comes entirely from the mooring lines' own
geometry (each fairlead offset rotates with platform yaw, recomputing the line
force each step). Three fairlead frames accept the mooring forces as
`forceML{1,2,3}[1..3]`. Wave force enters separately through `waveForces[1..3]`,
wave moment through `waveMoments[1..3]`.

I export the whole thing as a single FMI 2.0 co-simulation FMU with `omc`
(OpenModelica): `models/FOWT.fmu`, used for both sea states, since wind
and wave are fed in as external inputs rather than baked into the model. From
there, Python (`simulation/cosim_runner.py`, via FMPy's `FMU2Slave`) steps the
FMU at 0.02 s. The FMU embeds an implicit CVODE solver, so it sub-steps its own
stiff dynamics and this communication step only samples the platform response.
Here's the actual loop body. This *is* the fmu run + mooring coupling, line by
line:

In [3]:
import inspect

print(inspect.getsource(run_cosim))

def run_cosim(mooring_model, sea_state: str, duration: float = 600.0,
              dt: float = 0.02) -> dict:
    """Step the FOWT FMU for `sea_state`, computing 3 mooring line forces each step.

    The FMU embeds a CVODE internal solver, so the communication `dt` only needs
    to resolve the platform/wave dynamics; 0.02 is a good default.

    Wave elevation and wind speed are generated from the sea state parameters
    and fed into the FMU as external inputs each step. `mooring_model` must
    implement `.compute(horizontal_offset) -> dict` with keys
    `fairlead_tension_N`, `horizontal_force_N`, `vertical_force_N`.

    Collective pitch is fixed for the whole run at `pitch_schedule(env["U_hub"])`
    -- the sea state's mean wind speed's published steady operating point, not
    re-targeted every step -- so turbulence produces natural thrust variation
    instead of being cancelled by an idealized per-step controller.

    Returns dict of time series: time, surge, sway, heave, ro

This closes the feedback loop that a static benchmark cannot exercise. At step
*i*, each line's horizontal fairlead-anchor offset comes from the FMU state at
step *i-1*: surge and sway translate the fairlead directly, while roll, pitch
and yaw shift its horizontal position through the rotation matrix. The
resulting mooring forces are fed back to the FMU, and the new platform state
becomes the input to the next step.

Heave remains part of the 6DOF platform dynamics, but the Part 1 surrogate
consumes horizontal fairlead-anchor offset only, so instantaneous vertical-span
changes are not passed into the mooring model. For the heave amplitudes seen
here, that omission has a sub-percent effect on horizontal tension.

---

## Two Mooring Models, One Interchangeable Interface

Both mooring models implement the same `compute(horizontal_offset) -> dict`
interface, called once per line, three times per step:

- **`CatenaryMooring`**: Part 1's Newton catenary solve, used here as the
  analytical reference and capped at ~2100 kN at the 807.6 m taut limit.
- **`OnnxMooring`**: Part 1's cached MLP surrogate, clipped to the same taut
  cap so it never extrapolates past it.

Sweeping a single line's horizontal offset (geometry only, no FMU needed) shows
where they agree and where the surrogate starts to drift.

In [4]:
catenary = CatenaryMooring()
onnx = OnnxMooring(data_dir=ONNX_DATA_DIR)

offsets = np.linspace(750, 808, 200)
tension_cat = [catenary.compute(o)["fairlead_tension_N"] / 1e3 for o in offsets]
tension_onnx = [onnx.compute(o)["fairlead_tension_N"] / 1e3 for o in offsets]

fig_sweep = go.Figure()
fig_sweep.add_trace(go.Scatter(x=offsets, y=tension_cat, name="Catenary (analytical reference)"))
fig_sweep.add_trace(go.Scatter(x=offsets, y=tension_onnx, name="ONNX surrogate", line={"dash": "dot"}))
fig_sweep.update_layout(title="Single-line tension vs horizontal offset",
                         xaxis_title="Horizontal offset (m)", yaxis_title="Fairlead tension (kN)",
                         height=450)
fig_sweep.show()

![CatenaryMooring iterates a nonlinear solve; OnnxMooring is one MLP forward pass](figures/mooring_models_interface.png)

*Same interface, different insides. `CatenaryMooring` solves the nonlinear
catenary equations for `H` and `L_b`. `OnnxMooring` returns the same outputs
through the MLP frozen in Part 1.*

That swap happens in the Python loop, not inside the `.mo` model. Wolfram System
Modeler ships a built-in ONNX-import block, so the surrogate could sit inside the
Modelica diagram there. OpenModelica 1.24.4 has no equivalent, so I push the
boundary out to the FMU's edge instead: `FOWT.fmu` only ever sees three force
vectors over `forceML{1,2,3}`, and `cosim_runner.py` decides one level above
whether those came from a Newton solve or an MLP. The FMU never needs to know
which.

---

## Results in Both Sea States

Both sea states from the table above ran the full 200 s, each under both
mooring models. The Kaimal wind and JONSWAP wave time series actually driving
the FMU each step:

In [5]:
plot_environment(SEA_STATES, DURATION).show()

Now the real thing: for each sea state, step `FOWT.fmu` for a full 200 s run with
`CatenaryMooring` in the loop, then again with `OnnxMooring`, the same
`run_cosim` loop from Part 2, mooring force resolved fresh every step. Four FMU
runs, ~10,000 steps each:

In [6]:
results = {name: run_sea_state_timed(name) for name in SEA_STATES}
print("done:", list(results.keys()))

done: ['operational', 'storm']


In [7]:
summary_table(results["operational"])

Surge, mean (m)                     8.739885
Surge, max abs (m)                 12.979638
Heave, dynamic range (m)             0.06678
Roll, max abs (deg)                 0.310822
Pitch, max abs (deg)                1.398022
Yaw, max abs (deg)                   0.04378
Tension 1, settled range (kN)      714 - 924
Tension 1, raw peak (kN)                1153
Tension 2, settled range (kN)    1309 - 1598
Tension 2, raw peak (kN)                1598
Tension 3, settled range (kN)    1308 - 1597
Tension 3, raw peak (kN)                1597
dtype: object

In [8]:
plot_tension(results["operational"], "Mooring tension, all 3 lines - operational sea state").show()
plot_angles(results["operational"], "Heel / trim / yaw - operational sea state").show()

The operational case develops a substantial mean surge offset, consistent with
sustained rotor thrust, while pitch reflects the combined loading and
hydrostatic restoring. I do not isolate the individual load contributions here.
All three lines remain below the taut cap.

In [9]:
summary_table(results["storm"])

Surge, mean (m)                       4.6093
Surge, max abs (m)                  9.107746
Heave, dynamic range (m)             0.06323
Roll, max abs (deg)                 0.325924
Pitch, max abs (deg)                1.651292
Yaw, max abs (deg)                  0.057286
Tension 1, settled range (kN)     809 - 1164
Tension 1, raw peak (kN)                1164
Tension 2, settled range (kN)    1148 - 1438
Tension 2, raw peak (kN)                1438
Tension 3, settled range (kN)    1148 - 1437
Tension 3, raw peak (kN)                1437
dtype: object

In [10]:
plot_tension(results["storm"], "Mooring tension, all 3 lines - storm sea state").show()
plot_angles(results["storm"], "Heel / trim / yaw - storm sea state").show()

Mean surge is *lower* in the storm case, not higher, because mean rotor thrust
is lower: with the fixed Region-3 pitch schedule, the blades feather further at
20 m/s (17.47°) than at 12 m/s (3.83°) to hold power near rated, and thrust
falls off as a result -- the same behavior a real turbine shows past rated wind
speed. Turbulence and wave loading are still much stronger in the storm case,
and that shows up as thrust variability (std 156 kN against 54 kN operational)
and a redistribution of line loading -- line 1's settled range widens (210 kN
to 355 kN, +69%) while lines 2/3 stay essentially flat (289 kN to 290 kN,
+0%) -- rather than a uniformly wider tension range or a bigger mean offset.

Statistics in the summary tables above skip the first 20 s while the platform
settles from its initial condition. In the operational case, line 1's raw peak
(1153 kN) lands on the very first step and doesn't recur; in the storm case,
line 1's raw peak (1164 kN) is now a genuine settled-response maximum, not a
first-step artifact -- the higher thrust variability pushes it there. Lines 2
and 3 have no first-step spike in either sea state: their raw peaks (1598 kN
operational, 1437-1438 kN storm) are genuine settled-response maxima. Yaw
restoring comes from the mooring lines' own geometry, not a spring: its
reported 0.04-0.06° max is a real, if small, settled-response value, not a
placeholder artifact.

---

## So, What Did We Learn?

Part 1 only tested the surrogate against a frozen horizontal offset. Here its
force predictions are fed back into a moving platform, and that motion becomes
the next mooring input. Across both sea states, the mean absolute tension
difference stays near 10 kN and the maximum reaches 23-27 kN across sea states,
against line tensions in the 0.7-1.6 MN range across the settled window.

In [11]:
benchmark = benchmark_table(results)
benchmark

,runtime_catenary_s,runtime_onnx_s,mean_abs_error_kN,max_abs_error_kN
sea_state,,,,
operational,8.350994,6.564677,9.584282,23.530764
storm,8.823239,6.139060,11.559807,26.970483


The effect on the platform trajectory is concentrated in surge:

In [12]:
motion_diff_table(results)

,surge RMSE (m),surge max|d| (m),heave RMSE (m),heave max|d| (m),pitch RMSE (deg),pitch max|d| (deg)
sea_state,,,,,,
operational,0.218958,0.381135,0.005735,0.007978,0.016778,0.023073
storm,0.185232,0.526388,0.005643,0.007962,0.013499,0.028215


Surge RMSE is roughly 2-4% of each sea state's mean offset, so the difference
is measurable rather than numerical dust. Heave and pitch RMSE stay small,
~6 mm and ~0.014-0.018°. Sway stays under 7 mm. Roll and yaw differences stay
below 0.0025° in both cases.

The Part 2 takeaway. A surrogate trained against the quasi-static catenary
relation can sit inside a dynamic 6DOF platform feedback loop without the
approximation spreading equally into every platform motion. The largest effect
appears where the mooring matters most here: surge.

Both mooring models are still quasi-static. `compute(offset)` returns the
equilibrium tension for a given offset, with no line mass, inertia, or
added-mass dynamics of its own. Real chain has weight and drag that lag the
platform's motion. Snap loads and line resonance are therefore outside what
either the catenary model or a surrogate trained on it can produce.

---